# RouterV3 -- environment bring-up + end-to-end verification

Builds a headless bridge to KiCad's real interactive router (`PNS::ROUTER`) by compiling a small pybind11 module *inside* a KiCad 9.0.8 source checkout, then proves it actually works by routing a real net on a toy board and independently checking the result.

Works on Google Colab or any Linux Jupyter kernel with `sudo` (a local container, a cloud notebook VM, etc.) -- nothing below is Colab-specific except the optional Drive-caching note in step 3.

**Status: unverified.** The C++ in `pcbworld/engine/cpp/` was written from reading the KiCad 9.0.8 source directly (see `docs/engine_access.md` for citations) but has never been compiled. Expect this notebook to need iteration on a first run -- missing link targets, include path issues, CMake flags that don't exist on this KiCad version, etc. Run it cell by cell; each stage asserts before moving on, so it fails loudly and close to the actual problem rather than silently producing garbage.

Rough time budget: ~10 min installing build deps, ~30-60 min compiling KiCad + the bridge (first run only -- see the caching note in step 3 to skip this on future sessions).

Re-running this notebook in a later session? Just run step 1 (clone/pull) and it'll pick up anything I've since pushed -- no need to redo it from scratch.

## 0. Working directory
Everything below is relative to `WORKDIR`. Defaults to `/content` on Colab, `~/routerv3-build` elsewhere.

In [ ]:
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORKDIR = Path("/content") if IN_COLAB else Path.home() / "routerv3-build"
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

REPO_URL = "https://github.com/Klutzhehe/Routerv3.git"
KICAD_TAG = "9.0.8"

os.environ["WORKDIR"] = str(WORKDIR)
os.environ["REPO_URL"] = REPO_URL
os.environ["KICAD_TAG"] = KICAD_TAG
print(f"WORKDIR={WORKDIR}  IN_COLAB={IN_COLAB}")

## 2. System build dependencies
Uses KiCad's own PPA to pull the exact build-dependency list for 9.0, rather than hand-listing wxWidgets/Boost/OCCT/etc versions ourselves. Also installs the full `kicad` package itself, which gives us a working system `pcbnew` Python module -- used below purely as an independent test-fixture generator/checker, decoupled from the code we're testing.

In [ ]:
%%bash
set -e
sudo add-apt-repository -y ppa:kicad/kicad-9.0-releases

# `apt-get build-dep` needs deb-src entries, which are disabled by default
# both in Colab's base sources.list and in the PPA's own (auto-added,
# commented-out) deb-src line. Enable whatever's there commented-out...
sudo sed -i -E 's/^#\s*(deb-src .*)/\1/' /etc/apt/sources.list /etc/apt/sources.list.d/*.list || true
# ...and if the base image stripped deb-src out entirely rather than just
# commenting it, fall back to mirroring every deb line as deb-src.
if ! grep -rq '^deb-src ' /etc/apt/sources.list; then
  grep '^deb ' /etc/apt/sources.list | sed 's/^deb /deb-src /' | sudo tee -a /etc/apt/sources.list >/dev/null
fi

sudo apt-get update -qq
sudo apt-get build-dep -y kicad
sudo apt-get install -y kicad cmake ninja-build
# `apt-get build-dep` didn't pull these on Colab even though KiCad's
# CMakeLists.txt does a hard `find_package(OpenGL REQUIRED)` for the `gal`
# library we link against -- install explicitly rather than relying on
# build-dep to have covered it.
sudo apt-get install -y libgl1-mesa-dev libglu1-mesa-dev libglvnd-dev mesa-common-dev
pip install -q pybind11

## 1. System build dependencies
Uses KiCad's own PPA to pull the exact build-dependency list for 9.0, rather than hand-listing wxWidgets/Boost/OCCT/etc versions ourselves. Also installs the full `kicad` package itself, which gives us a working system `pcbnew` Python module -- used below purely as an independent test-fixture generator/checker, decoupled from the code we're testing.

In [ ]:
## 3. Clone KiCad source
If you want to skip the ~30-60 min build on future sessions, copy `$WORKDIR/kicad-src/build` to Drive after a successful build and restore it here before step 5 instead of re-running cmake/ninja from scratch.

In [ ]:
%%bash
set -e
cd "$WORKDIR"
if [ ! -d kicad-src ]; then
  git clone --depth 1 --branch "$KICAD_TAG" https://gitlab.com/kicad/code/kicad.git kicad-src
fi

## 4. Wire the bridge into KiCad's build
Copies `pcbworld/engine/cpp` into the KiCad checkout and appends an `add_subdirectory()` to `pcbnew/CMakeLists.txt` so it's configured *after* the `pnsrouter`/`pcbcommon`/etc targets it links against are defined.

In [ ]:
%%bash
set -e
cd "$WORKDIR"
if [ ! -d Routerv3 ]; then git clone "$REPO_URL" Routerv3; else (cd Routerv3 && git pull); fi
if [ ! -d kicad-src ]; then
  git clone --depth 1 --branch "$KICAD_TAG" https://gitlab.com/kicad/code/kicad.git kicad-src
fi

## 5. Configure + build
Release build (not Debug) -- see `docs/performance.md`, a debug KiCad build is much slower and would waste the whole point of this being a fast headless engine. Only builds the `pcbworld_pns_bridge` target and its dependencies, not the full GUI app.

In [ ]:
%%bash
set -e
cd "$WORKDIR/kicad-src"

# Skip reconfiguring if a previous run already succeeded (e.g. re-running
# this notebook against a build dir restored from Drive).
if [ ! -f build/CMakeCache.txt ]; then
  cmake -S . -B build -G Ninja \
    -DCMAKE_BUILD_TYPE=Release \
    -DKICAD_BUILD_QA_TESTS=OFF \
    -DKICAD_SCRIPTING_WXPYTHON=OFF \
    -DKICAD_BUILD_I18N=OFF
fi

## 4. Configure + build
Release build (not Debug) -- see `docs/performance.md`, a debug KiCad build is much slower and would waste the whole point of this being a fast headless engine. Only builds the `pcbworld_pns_bridge` target and its dependencies, not the full GUI app.

In [ ]:
## 6. Import the compiled bridge

In [ ]:
%%bash
set -e
cd "$WORKDIR/kicad-src"
cmake --build build --target pcbworld_pns_bridge -j"$(nproc)"

## 7. End-to-end verification: route a real net on a toy board

This is the actual proof the whole stack works, not just "it imported":

1. Generate a 2-pad, 1-net board with the **system** `pcbnew` module
   (`scripts/make_toy_board.py`) -- independent of the bridge we're testing.
2. Load it with our bridge, find both pads via `query_hover_items`, route
   between them, commit, save.
3. Reload the saved file with the **system** `pcbnew` module again and
   assert a track actually exists connecting the two pads' net. This
   catches the failure mode where our bridge *thinks* it committed a route
   but the board file on disk doesn't actually reflect it.

In [ ]:
import sys, glob

matches = glob.glob(f"{WORKDIR}/kicad-src/build/**/pcbworld_pns_bridge*.so", recursive=True)
assert matches, "build didn't produce the extension -- check step 4 output"
sys.path.insert(0, str(Path(matches[0]).parent))

import pcbworld_pns_bridge as bridge

print(bridge.__file__)
print("bridge module loaded OK")

## 6. End-to-end verification: route a real net on a toy board

This is the actual proof the whole stack works, not just "it imported":

1. Generate a 2-pad, 1-net board with the **system** `pcbnew` module
   (`scripts/make_toy_board.py`) -- independent of the bridge we're testing.
2. Load it with our bridge, find both pads via `query_hover_items`, route
   between them, commit, save.
3. Reload the saved file with the **system** `pcbnew` module again and
   assert a track actually exists connecting the two pads' net. This
   catches the failure mode where our bridge *thinks* it committed a route
   but the board file on disk doesn't actually reflect it.

In [ ]:
%%bash
set -e
cd "$WORKDIR"
python3 Routerv3/scripts/make_toy_board.py toy_board.kicad_pcb

In [ ]:
board_path = str(WORKDIR / "toy_board.kicad_pcb")

b = bridge.PNSBridge()
assert b.load_board(board_path), "load_board failed"

nets = b.net_names()
print("nets:", nets)
assert "toy_net" in nets

# Pad positions match scripts/make_toy_board.py: (0, 0) mm and (20, 0) mm.
MM = 1_000_000  # KiCad internal units are nm
pad_a = b.query_hover_items(0, 0, layer=0, slop_radius=int(0.5 * MM))
pad_b = b.query_hover_items(20 * MM, 0, layer=0, slop_radius=int(0.5 * MM))
print("pad_a candidates:", [(c.kind, c.x, c.y) for c in pad_a])
print("pad_b candidates:", [(c.kind, c.x, c.y) for c in pad_b])
assert pad_a and pad_b, "couldn't find both pads via query_hover_items"

b.set_mode(bridge.MODE_ROUTE_SINGLE)
b.set_track_width(int(0.25 * MM))

assert b.start_route(pad_a[0].x, pad_a[0].y, pad_a[0].id, 0), "start_route failed"
assert b.push(10 * MM, 0, -1), "push failed"
assert b.fix(pad_b[0].x, pad_b[0].y, pad_b[0].id, False, False), "fix failed"
b.commit_routing()
assert b.save_board(board_path), "save_board failed"
print("routed and saved OK")

In [ ]:
# Independent check: reload with the system pcbnew module (not our bridge)
# and confirm a real track exists on the net.
check_board = pcbnew.LoadBoard(board_path)
tracks = list(check_board.GetTracks())
toy_net_tracks = [t for t in tracks if t.GetNetname() == "toy_net"]

print(f"{len(tracks)} total track/via items, {len(toy_net_tracks)} on toy_net")
assert toy_net_tracks, "no track found on toy_net after routing -- commit didn't reach disk"
print("VERIFIED: PNS::ROUTER routed a real track headlessly and it round-tripped to disk.")